In [1]:
from importlib.metadata import version

used_libraries = {
    "reasoning_from_scratch": version("reasoning_from_scratch"),
    "torch": version("torch"),
    "numpy": version("numpy"),
    "sympy": version("sympy"),
    "tokenizers": version("tokenizers")
}
print(used_libraries)

{'reasoning_from_scratch': '0.2.0', 'torch': '2.10.0', 'numpy': '2.5.3', 'sympy': '1.14.0', 'tokenizers': '0.23.2'}


In [2]:
from pathlib import Path
import torch

from reasoning_from_scratch.qwen3 import (Qwen3Model, Qwen3Tokenizer, QWEN_CONFIG_06_B)


In [3]:
def load_model_and_tokenizer():
    model_path = Path("./qwen3_model/") / "qwen3-0.6B-base.pth"
    tokenizer_path = Path("./qwen3_model/") / "tokenizer-base.json"

    tokenizer = Qwen3Tokenizer(tokenizer_path)
    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()

In [4]:
from reasoning_from_scratch.qwen3 import KVCache

def generate_stream_with_cache(model, input_tokens, max_new_tokens, eos_token_id=None):
    cache = KVCache(n_layers = model.cfg['n_layers'])
    model.reset_kv_cache()
    out = model(input_tokens, cache=cache)[:,-1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break

        yield next_token
        input_tokens = next_token
        out = model(input_tokens, cache=cache)[:,-1]


In [5]:
#device = torch.device("mps")
device = torch.device("cpu")

In [6]:
prompt = r"If $a+b=3$ and $ab=\t\frac{13}{6}$, what are the values of $a$ and $b$?"

In [7]:
input_tokens = tokenizer.encode(prompt)
input_tokens = torch.tensor(input_tokens).unsqueeze(0)

In [11]:
all_token_ids = []
for token in generate_stream_with_cache(model, input_tokens, max_new_tokens=512, eos_token_id=tokenizer.eos_token_id):
    token_id = token.squeeze(0)
    decoded_token = tokenizer.decode(token_id.tolist())
    print(decoded_token, end='', flush=True)
    all_token_ids.append(token_id)

all_tokens = tokenizer.decode(all_token_ids)
#print(all_tokens)

 Express your answer as a common fraction. To find the values of \(a\) and \(b\) given that \(a + b = 3\) and \(ab = \frac{13}{6}\), we can use the properties of quadratic equations. Specifically, if \(a\) and \(b\) are roots of the quadratic equation \(x^2 - (a+b)x + ab = 0\), then the equation becomes:

\[x^2 - 3x + \frac{13}{6} = 0\]

We can solve this quadratic equation using the quadratic formula \(x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}\), where \(a = 1\), \(b = -3\), and \(c = \frac{13}{6}\). Let's calculate the discriminant first:

\[
b^2 - 4ac = (-3)^2 - 4 \cdot 1 \cdot \frac{13}{6} = 9 - \frac{52}{6} = 9 - \frac{26}{3} = \frac{27}{3} - \frac{26}{3} = \frac{1}{3}
\]

So the solutions are:

\[
x = \frac{3 \pm \sqrt{\frac{1}{3}}}{2} = \frac{3 \pm \frac{\sqrt{3}}{3}}{2} = \frac{9 \pm \sqrt{3}}{6}
\]

Thus, the values of \(a\) and \(b\) are \(\frac{9 + \sqrt{3}}{6}\) and \(\frac{9 - \sqrt{3}}{6}\). Let's verify this using Python code.
```python
from sympy import symbols, Eq, solve,

In [12]:
from IPython.display import Latex, display
display(Latex(all_tokens))

<IPython.core.display.Latex object>